In [ ]:
# ===================== CONFIG =====================
MODEL_NAME    = "my_voice_model"
TARGET_DIR   = "/kaggle/input/datasets/MY-DATASET/target"
MY_VOICE_DIR = "/kaggle/input/datasets/MY-DATASET/my_voice"  # optional: your voice, or None

SAMPLE_RATE   = "40000"   # "32000" | "40000" | "48000"  (40k is the standard choice)
TOTAL_EPOCHS  = 150       # ~200-400 for 10-30 min of audio; fewer for big datasets
SAVE_EVERY    = 10        # checkpoint + exported .pth every N epochs
BATCH_SIZE    = 8
GPUS          = "0"       # "0-1" uses both T4s (faster, but multi-GPU is less battle-tested)

# Silence stripping
SILENCE_TOP_DB = 40       # anything this many dB below the file's peak counts as silence. Raise it (45-50) if quiet words get cut
KEEP_PAUSE_S   = 0.25     # every long gap gets shortened to this length

# Resume: to continue a run that hit the 12 h limit, add that version's output as an input and point this to
# .../rvc_logs/<MODEL_NAME>. TOTAL_EPOCHS must be larger than the epoch it reached.
RESUME_DIR    = None
# ==================================================

import os
print("Inputs available:")
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.count(os.sep) - 2
    if depth <= 2:
        print("  " * depth + os.path.basename(root) + "/", f"({len(files)} files)" if files else "")

# RVC voice model training (Kaggle)

This notebook uses [Applio](https://github.com/IAHispano/Applio) 3.7.1 (an RVC v2 fork) to train a voice model.

**Kaggle setup**
1. In the right sidebar, open *Settings*. Set **Accelerator to GPU T4 x2** (recent PyTorch builds no longer run on the P100) and turn **Internet on**.
2. Upload your audio as a Kaggle Dataset (*Add Input → Upload*). Use this layout:
   ```
   target/     <- the voice you want to sound like (required). Any format: wav/mp3/flac/m4a/...
   my_voice/   <- your own voice (optional). Used only to pick the pitch shift and to test the result
   ```
3. Edit the config cell, then run *Save Version → Save & Run All (Commit)*. The run continues after you close the tab (up to 12 h).
   Everything you need is saved to `/kaggle/working`, so it appears under the version's *Output* tab.

**How much audio:** 10–30 minutes of clean speech is the sweet spot. Under 5 minutes works poorly.
The silence-stripping step below reports how much actual speech you have.

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    print(root, f"({len(files)} files)" if files else "")

## 1. Install Applio

**Dependency note:** Kaggle's kernel already has NumPy loaded in memory when this notebook starts. Replacing NumPy/SciPy on disk while the kernel is running leaves a mix of old (in memory) and new (on disk) files, which causes errors like `cannot import name '_slice' from 'numpy._core.umath'`. So the setup cell **does not touch NumPy, SciPy, torch or torchaudio**. It installs Applio's other requirements with Kaggle's current versions of those packages held fixed using a pip constraints file. Do not `pip install` NumPy/SciPy anywhere else in this notebook.


In [ ]:
import subprocess, sys, os, shutil
from importlib.metadata import version, PackageNotFoundError
shutil.rmtree("/tmp/rvc", ignore_errors=True)

APPLIO = "/tmp/rvc/Applio"          # outside /kaggle/working so the repo and models don't bloat the output
LOGS   = "/kaggle/working/rvc_logs"     # training output lives here, so it's saved even if the run gets cut off
os.makedirs("/tmp/rvc", exist_ok=True)

def run(cmd, cwd=APPLIO):
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed with exit code {p.returncode}")

if not os.path.isdir(APPLIO):
    run(["git", "clone", "--depth", "1", "--branch", "3.7.1", "https://github.com/IAHispano/Applio.git", APPLIO], cwd="/kaggle")

# Packages we must NOT change while this kernel is running (numpy is already loaded in memory).
# Freeze whatever Kaggle has installed and tell pip to keep it.
HOLD = ("numpy", "scipy", "torch", "torchaudio")
held = []
for pkg in HOLD:
    try:
        held.append(f"{pkg}=={version(pkg)}")
    except PackageNotFoundError:
        pass
open("/tmp/rvc/constraints.txt", "w").write("\n".join(held) + "\n")
print("holding:", held)

# Drop those packages from Applio's requirements; everything else installs normally
reqs = [l for l in open(f"{APPLIO}/requirements-engine.txt").read().splitlines()
        if l.split("==")[0].split(">")[0].split("<")[0].split(";")[0].strip().lower() not in HOLD]
open("/tmp/rvc/req.txt", "w").write("\n".join(reqs))
run([sys.executable, "-m", "pip", "install", "-q", "-r", "/tmp/rvc/req.txt", "-c", "/tmp/rvc/constraints.txt"], cwd="/kaggle")

# Verify in a fresh Python process that nothing got broken
run([sys.executable, "-c",
     "import numpy, scipy, scipy.signal, librosa; "
     "print('NumPy:', numpy.__version__); "
     "print('SciPy:', scipy.__version__); "
     "print('librosa:', librosa.__version__); "
     "print('imports OK')"], cwd="/kaggle")

# Point Applio's logs dir at /kaggle/working. Copy the bundled mute/reference files it expects to find there
shutil.copytree(f"{APPLIO}/logs", LOGS, dirs_exist_ok=True)
os.environ["APPLIO_LOGS_DIR"] = LOGS

# Pretrained HiFi-GAN generator/discriminator, contentvec embedder, rmvpe pitch model
run([sys.executable, "core.py", "prerequisites", "--pretraineds-hifigan", "--models", "--no-exe"])

import torch
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


In [ ]:
import sys
from importlib.metadata import version
import numpy as np
import scipy

print("Python:", sys.executable)
print("NumPy in memory:", np.__version__, "| on disk:", version("numpy"))
print("SciPy in memory:", scipy.__version__, "| on disk:", version("scipy"))
if np.__version__ != version("numpy") or scipy.__version__ != version("scipy"):
    print("MISMATCH: packages changed on disk after the kernel loaded them. Restart the session and run again.")

try:
    import scipy.signal, librosa
    print("scipy.signal + librosa: OK")
except Exception as e:
    print("import FAILED:", repr(e))


## 2. Strip silence and gaps

RVC trains on short chunks. Chunks that are mostly silence waste training time and teach the model that silence is part of the voice.
Applio's slicer already cuts at silences, but it keeps a fair amount of dead air. This step first shortens every gap
to `KEEP_PAUSE_S` and drops clicks or blips shorter than 0.3 s. It also converts all formats to mono WAV.

In [ ]:
import glob, librosa, numpy as np, soundfile as sf

AUDIO_EXT = (".wav", ".mp3", ".flac", ".m4a", ".ogg", ".opus", ".aac", ".wma", ".webm", ".mp4")

def audio_files(d):
    return sorted(p for p in glob.glob(os.path.join(d, "**", "*"), recursive=True) if p.lower().endswith(AUDIO_EXT))

def strip_silence(src_dir, dst_dir):
    os.makedirs(dst_dir, exist_ok=True)
    total_in = total_out = 0.0
    files = audio_files(src_dir)
    if not files:
        raise FileNotFoundError(f"No audio files found in {src_dir}")
    for i, path in enumerate(files):
        y, sr = librosa.load(path, sr=None, mono=True)
        pad, gap = int(0.08 * sr), np.zeros(int(KEEP_PAUSE_S * sr), dtype=y.dtype)
        pieces = []
        for s, e in librosa.effects.split(y, top_db=SILENCE_TOP_DB, frame_length=2048, hop_length=512):
            if e - s < 0.3 * sr:   # clicks, breaths, blips
                continue
            pieces += [y[max(0, s - pad):min(len(y), e + pad)], gap]
        total_in += len(y) / sr
        if not pieces:
            print(f"  skipped (no speech found): {os.path.basename(path)}")
            continue
        out = np.concatenate(pieces)
        total_out += len(out) / sr
        sf.write(os.path.join(dst_dir, f"{i:04d}.wav"), out, sr, subtype="PCM_16")
    print(f"{src_dir}: {len(files)} files, {total_in/60:.1f} min -> {total_out/60:.1f} min of speech "
          f"({100 * (1 - total_out / max(total_in, 1e-9)):.0f}% removed)")
    return total_out

CLEAN_TARGET = "/tmp/rvc/clean/target"
CLEAN_MINE   = "/tmp/rvc/clean/my_voice"
shutil.rmtree("/tmp/rvc/clean", ignore_errors=True)

speech_s = strip_silence(TARGET_DIR, CLEAN_TARGET)
if speech_s < 5 * 60:
    print("WARNING: under 5 min of speech. Expect a weak model. Aim for 10+ min.")

HAVE_MINE = bool(MY_VOICE_DIR) and os.path.isdir(MY_VOICE_DIR) and audio_files(MY_VOICE_DIR)
if HAVE_MINE:
    strip_silence(MY_VOICE_DIR, CLEAN_MINE)

# Listen to a sample to make sure nothing important got cut
from IPython.display import Audio, display
y, sr = librosa.load(sorted(glob.glob(f"{CLEAN_TARGET}/*.wav"))[0], sr=None, duration=15)
display(Audio(y, rate=sr))

## 3. (Optional) Pick a pitch shift from your own voice

Your voice is **not** used for training. RVC learns only the target voice, and the content encoder handles any input speaker.
Your voice is still useful here: comparing the median pitch of the two voices tells you how many semitones to shift at
conversion time. Male→female is typically around +12, and female→male around −12.

In [ ]:
def median_f0(clean_dir, seconds=40):
    ys = []
    for p in sorted(glob.glob(f"{clean_dir}/*.wav")):
        y, _ = librosa.load(p, sr=16000, mono=True)
        ys.append(y)
        if sum(map(len, ys)) > seconds * 16000:
            break
    y = np.concatenate(ys)[: seconds * 16000]
    f0, voiced, _ = librosa.pyin(y, fmin=60, fmax=900, sr=16000, frame_length=1024)
    return float(np.nanmedian(f0[voiced]))

PITCH_SHIFT = 0
f0_target = median_f0(CLEAN_TARGET)
print(f"target voice median pitch: {f0_target:.0f} Hz")
if HAVE_MINE:
    f0_mine = median_f0(CLEAN_MINE)
    PITCH_SHIFT = int(round(12 * np.log2(f0_target / f0_mine)))
    print(f"your voice median pitch:   {f0_mine:.0f} Hz")
    print(f"=> suggested pitch shift when converting your voice: {PITCH_SHIFT:+d} semitones")

## 4. Preprocess and extract features (pitch via RMVPE, content via ContentVec)

In [ ]:
CPU = str(min(os.cpu_count() or 2, 64))

run([sys.executable, "core.py", "preprocess", "--model-name", MODEL_NAME, "--dataset-path", CLEAN_TARGET,
     "--sample-rate", SAMPLE_RATE, "--cpu-cores", CPU, "--cut-preprocess", "Automatic",
     "--chunk-len", "3.0", "--overlap-len", "0.3", "--normalization-mode", "post"])

run([sys.executable, "core.py", "extract", "--model-name", MODEL_NAME, "--f0-method", "rmvpe",
     "--cpu-cores", CPU, "--gpu", "0", "--sample-rate", SAMPLE_RATE,
     "--embedder-model", "contentvec", "--include-mutes", "2"])

## 5. Train

Progress prints once per epoch. Every `SAVE_EVERY` epochs, a usable model is written to `rvc_logs/<MODEL_NAME>/<MODEL_NAME>_<epoch>e_<step>s.pth`.
The faiss index is built automatically when training finishes.

In [ ]:
model_dir = f"{LOGS}/{MODEL_NAME}"

if RESUME_DIR:
    # Restore the G/D training checkpoints. train.py resumes from the latest one it finds
    for p in glob.glob(f"{RESUME_DIR}/[GD]_*.pth"):
        shutil.copy(p, model_dir)
        print("restored", os.path.basename(p))

run([sys.executable, "core.py", "train", "--model-name", MODEL_NAME,
     "--sample-rate", SAMPLE_RATE, "--total-epoch", str(TOTAL_EPOCHS), "--save-every-epoch", str(SAVE_EVERY),
     "--batch-size", str(BATCH_SIZE), "--gpu", GPUS, "--vocoder", "HiFi-GAN",
     "--pretrained", "--save-only-latest", "--index-algorithm", "Auto"])
# NOTE: do NOT add --save-every-weights. It is a Click flag with default=True, so passing it turns it OFF.

## 6. Collect the final model (.pth + .index) into `/kaggle/working/final_model.zip`

In [ ]:
import re

def epoch_of(p):
    m = re.search(r"_(\d+)e_(\d+)s\.pth$", p)
    return (int(m.group(1)), int(m.group(2))) if m else (-1, -1)

weights = sorted(glob.glob(f"{model_dir}/{MODEL_NAME}_*e_*s.pth"), key=epoch_of)
indexes = glob.glob(f"{model_dir}/*.index")
assert weights, f"No exported weights in {model_dir}. Check the training log above"
if not indexes:
    run([sys.executable, "core.py", "index", "--model-name", MODEL_NAME, "--index-algorithm", "Auto"])
    indexes = glob.glob(f"{model_dir}/*.index")

PTH, INDEX = weights[-1], indexes[0]
final = "/kaggle/working/final_model"
shutil.rmtree(final, ignore_errors=True); os.makedirs(final)
shutil.copy(PTH, final); shutil.copy(INDEX, final)
shutil.make_archive(final, "zip", final)
print("model:", os.path.basename(PTH), "| index:", os.path.basename(INDEX))
print("all saved epochs:", [os.path.basename(w) for w in weights])
print("download: /kaggle/working/final_model.zip")

## 7. Test: convert your own voice

Converts 30 s of your voice, or of the target audio if you didn't provide yours. The output should sound like the target speaker.
If it sounds metallic or robotic, try an earlier epoch from `rvc_logs/` (it may be overtrained) or lower `index_rate`.

In [ ]:
src = sorted(glob.glob(f"{CLEAN_MINE if HAVE_MINE else CLEAN_TARGET}/*.wav"))[0]
y, sr = librosa.load(src, sr=None, duration=30)
sf.write("/tmp/rvc/test_in.wav", y, sr)
out = "/kaggle/working/test_converted.wav"

run([sys.executable, "core.py", "infer", "--input-path", "/tmp/rvc/test_in.wav", "--output-path", out,
     "--pth-path", PTH, "--index-path", INDEX, "--pitch", str(PITCH_SHIFT if HAVE_MINE else 0),
     "--f0-method", "rmvpe", "--index-rate", "0.5", "--protect", "0.33"])

print("input:");  display(Audio("/tmp/rvc/test_in.wav"))
print("output:"); display(Audio(out))